In [1]:
import numpy as np
import scipy as sp
import suite2p as s2p
import TwoPUtils as tpu
import reward_relative as rrel
import os
import torch

%matplotlib inline

%load_ext autoreload
%autoreload 2

In [2]:
from reward_relative.path_dict_seahorse import path_dictionary as path_dict
# options: path_dict_josquin, path_dict_msosamac, etc.
path_dict

{'preprocessed_root': '/data/2p_data',
 'sbx_root': '/media/sosalab/T7/2p_raw_data',
 'gdrive_root': '/mnt/gdrive/2P_Data',
 'VR_Data': '/data/2p_data/VR_Data',
 'git_repo_root': '/home/sosalab/local_repos',
 'TwoPUtils': '/home/sosalab/local_repos/TwoPUtils',
 'home': '/home/sosalab',
 'fig_dir': '/data/2p_data/fig_scratch'}

In [3]:
mouse = "5433_03"
basedir = os.path.join(path_dict['preprocessed_root'],mouse)
sbxdir = os.path.join(path_dict['sbx_root'],mouse) 

custom_nplanes = 1 # if needed to override empty optotune params

basedir, sbxdir

('/data/2p_data/5433_03', '/media/sosalab/T7/2p_raw_data/5433_03')

In [4]:
file_list = rrel.sessions_dict.single_plane[mouse]

In [5]:
# ut.get_ind_of_exp_day(da.sessions_dict.single_plane, mouse, d) for d in [5,8,10]

In [17]:
file_list[10:11]

({'date': '30_12_2024',
  'scene': 'Env2_LocationB',
  'session': 1,
  'scan': 2,
  'exp_day': 10,
  'GD': 13},)

In [7]:
s2p.version

'1.0.0.1'

In [8]:
#these are the default settings as of early 4/2026, so i'm 
# just gonna pull them out here in perpetuity considering
#the rate at which things change for suite2p
settings = {'torch_device': 'cuda',
 'tau': 1.0, # change
 'fs': 10.0, # change
 'diameter': [12.0, 12.0], # change
 'run': {'do_registration': 1,
  'do_regmetrics': True,
  'do_detection': True,
  'do_deconvolution': True,
  'multiplane_parallel': False},
 'io': {'combined': True,
  'save_mat': False,
  'save_NWB': False,
  'save_ops_orig': True,
  'delete_bin': False,
  'move_bin': False},
 'registration': {'align_by_chan2': False,
  'nimg_init': 400, # change
  'maxregshift': 0.1, # change
  'do_bidiphase': False,
  'bidiphase': 0.0,
  'batch_size': 100,
  'nonrigid': True,
  'maxregshiftNR': 5,
  'block_size': (128, 128),
  'smooth_sigma_time': 0,
  'smooth_sigma': 1.15,
  'spatial_taper': 3.45,
  'th_badframes': 1.0,
  'norm_frames': True,
  'snr_thresh': 1.2,
  'subpixel': 10,
  'two_step_registration': False,
  'reg_tif': False,
  'reg_tif_chan2': False},
 'detection': {'algorithm': 'sparsery',
  'denoise': False,
  'block_size': (64, 64),
  'nbins': 5000,
  'bin_size': None,
  'highpass_time': 100,
  'threshold_scaling': 1.0,
  'npix_norm_min': 0.0,
  'npix_norm_max': 100,
  'max_overlap': 0.75,
  'soma_crop': True,
  'chan2_threshold': 0.25,
  'cellpose_chan2': False,
  'sparsery_settings': {'highpass_neuropil': 25,
   'max_ROIs': 5000,
   'spatial_scale': 0,
   'active_percentile': 0.0},
  'sourcery_settings': {'connected': True,
   'max_iterations': 20,
   'smooth_masks': False},
  'cellpose_settings': {'cellpose_model': 'cpsam',
   'img': 'max_proj / meanImg',
   'highpass_spatial': 0,
   'flow_threshold': 0.4,
   'cellprob_threshold': 0.0,
   'params': None,
   'params_chan2': None}},
 'classification': {'classifier_path': None,
  'use_builtin_classifier': False,
  'preclassify': 0.0},
 'extraction': {'snr_threshold': 0.0,
  'batch_size': 500,
  'neuropil_extract': True,
  'neuropil_coefficient': 0.7,
  'inner_neuropil_radius': 2,
  'min_neuropil_pixels': 350,
  'lam_percentile': 50.0,
  'allow_overlap': False,
  'circular_neuropil': False},
 'dcnv_preprocess': {'baseline': 'maximin',
  'win_baseline': 60.0,
  'sig_baseline': 10.0,
  'prctile_baseline': 8.0},
 'version': '1.0.0.1'}

In [9]:
def update_settings(k, v, search_dict, loc = 'top'):

    if k in search_dict:
            old_val = search_dict[k]
            search_dict[k] = v
            return {"incoming key":k, "dict level":loc, 'old_value':old_val, 'new_value':search_dict[k]}
    else:
        subdicts = [key for key, subd in search_dict.items() if isinstance( subd, dict)]

        bottom = True if len(subdicts) == 0 else False
        
        if bottom:
            if k in search_dict:
                old_val = search_dict[k]
                search_dict[k] = v
                return {"incoming key":k, "dict level":loc, 'old_value':old_val, 'new_value':search_dict[k]}
        else:
            for sub_dict_key in subdicts:
                ret =  update_settings(k, v, search_dict[sub_dict_key], loc = sub_dict_key)
                if not 'found' in ret:
                    return ret
        
    return {'incoming key':k, 'found':False}

In [10]:
update_settings('inner_neuropil', 0.7, settings)

{'incoming key': 'inner_neuropil', 'found': False}

In [11]:
two_p_utils_defaults = tpu.s2p.default_ops()
for k, v in two_p_utils_defaults.items():
    print(update_settings(k, v, settings))

{'incoming key': 'look_one_level_down', 'found': False}
{'incoming key': 'fast_disk', 'found': False}
{'incoming key': 'delete_bin', 'dict level': 'io', 'old_value': False, 'new_value': False}
{'incoming key': 'mesoscan', 'found': False}
{'incoming key': 'h5py', 'found': False}
{'incoming key': 'h5py_key', 'found': False}
{'incoming key': 'save_path0', 'found': False}
{'incoming key': 'subfolders', 'found': False}
{'incoming key': 'data_path', 'found': False}
{'incoming key': 'nplanes', 'found': False}
{'incoming key': 'nchannels', 'found': False}
{'incoming key': 'functional_chan', 'found': False}
{'incoming key': 'tau', 'dict level': 'top', 'old_value': 1.0, 'new_value': 1.0}
{'incoming key': 'fs', 'dict level': 'top', 'old_value': 10.0, 'new_value': 15.4609}
{'incoming key': 'force_sktiff', 'found': False}
{'incoming key': 'preclassify', 'dict level': 'classification', 'old_value': 0.0, 'new_value': 0}
{'incoming key': 'save_mat', 'dict level': 'io', 'old_value': False, 'new_value':

In [12]:
# ops = tpu.s2p.set_ops(d={
#                                'fast_disk':[],
#                                'delete_bin':False,
#                                'two_step_registration':True,
#                                'maxregshiftNR':10,
#                                'nchannels':1,
#                                'tau': 0.7, #0.65s for gcamp7 in Terada paper
#                                'functional_chan':1,
#                                'nimg_init': 2000,
                               
#                                'roidetect':True,
#                                'input_format':"h5",
#                                'h5py_key':'data',
#                                'sparse_mode':True,
#                                'detection': {'threshold_scaling':0.8},
#                                 })

    

In [18]:
for fn,f in enumerate(file_list[10:11]):

    fullpath = os.path.join(basedir,f['date'],f['scene'],"%s_%03d_%03d" % (f['scene'], f['session'], f['scan']))
    scanpath = os.path.join(sbxdir,f['date'],f['scene'],"%s_%03d_%03d" % (f['scene'], f['session'], f['scan']))
    h5path = os.path.join(basedir,f['date'],f['scene'],"%s_%03d_%03d.h5" % (f['scene'], f['session'], f['scan']))






    # scanmat, sbxfile = scanpath+'.mat', scanpath+'.sbx'
    scanmat, sbxfile = scanpath+'.mat', scanpath+'.sbx'
    info = tpu.scanner_tools.sbx_utils.loadmat(scanmat, sbx_version=3)

    mari_default_settings = {
                               
                               'delete_bin':False,
                               'two_step_registration':True,
                               'maxregshiftNR':10,
                               'nchannels':1,
                               'tau': 0.7, #0.65s for gcamp7 in Terada paper
                               
                               'nimg_init': 2000,
                               'fs':info['frame_rate'],
                               'roidetect':True,
                               'input_format':"h5",
                               'h5py_key':'data',
                               'sparse_mode':True,
                               'threshold_scaling':0.8,
                                }
    
    for k, v in mari_default_settings.items():
        print(update_settings(k, v, settings))



    if len(info['etl_table'])>0:
        nplanes = info['etl_table'].shape[0]
    else:
        nplanes = custom_nplanes
        print('"etl_table" was empty; hardcoding %d planes' % nplanes)

    print('nplanes=',nplanes)

    if not os.path.isfile(h5path):
        h5name = tpu.scanner_tools.sbx_utils.sbx2h5(scanpath,output_name=h5path, sbx_version=3)
    else:
        h5name = os.path.split(h5path)[-1]


    db = {
        'functional_chan':1,
        'fast_disk':[],
        "data_path": [os.path.split(fullpath)[0]], # Directory where your input files are located
        "save_path0": fullpath, # Directory where you want suite2p to write output files.
        "file_list": [h5name], # Specify files you'd like to specifically use in the data_path
        "input_format": "h5",
        'nplanes':nplanes, # each tiff has these many planes in sequence
        "nchannels": 1, # each tiff has these many channels per plane
        "keep_movie_raw": True,
        "batch_size": 500, # we will decrease the batch_size in case low RAM on computer
    }
    


    outputs = s2p.run_s2p(settings=settings, db = db)

    !rm {h5name} 

{'incoming key': 'delete_bin', 'dict level': 'io', 'old_value': False, 'new_value': False}
{'incoming key': 'two_step_registration', 'dict level': 'registration', 'old_value': True, 'new_value': True}
{'incoming key': 'maxregshiftNR', 'dict level': 'registration', 'old_value': 10, 'new_value': 10}
{'incoming key': 'nchannels', 'found': False}
{'incoming key': 'tau', 'dict level': 'top', 'old_value': 0.25, 'new_value': 0.7}
{'incoming key': 'nimg_init', 'dict level': 'registration', 'old_value': 2000, 'new_value': 2000}
{'incoming key': 'fs', 'dict level': 'top', 'old_value': 15.625, 'new_value': 15.625}
{'incoming key': 'roidetect', 'found': False}
{'incoming key': 'input_format', 'found': False}
{'incoming key': 'h5py_key', 'found': False}
{'incoming key': 'sparse_mode', 'found': False}
{'incoming key': 'threshold_scaling', 'dict level': 'detection', 'old_value': 0.8, 'new_value': 0.8}
"etl_table" was empty; hardcoding 1 planes
nplanes= 1
dset size <HDF5 dataset "data": shape (25592, 